In [1]:
%matplotlib inline
# Cell 1 — parameters
LAT        = 16.8167    # Timbuktu (same as step1)
LON        = -2.9833
RADIUS_KM  = 100
LEVEL      = '06'
EPSILON    = 0.001

# Block 7 temporal parameters
PRIMARY_FROM = 1100
PRIMARY_TO   = 1200
WIDE_FROM    = 1000
WIDE_TO      = 2000
SINGLE_YEAR  = 1150     # step 7.4 single-step demo

# Honesty diagnostic threshold (provisional; step 7.3 will record it)
ECC_THRESHOLD = 10

In [2]:
# Cell 2 — imports and connection
import sys
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from shapely import wkt as shapely_wkt
from pathlib import Path
from IPython.display import Image as IPImage, display

sys.path.insert(0, '../../../..')
from scripts.shared.db_utils import db_connect
import scripts.shared.db_utils as _dbu

conn = db_connect()
print('Connected:', conn.execute('SELECT current_database()').fetchone())

ROOT = Path(_dbu.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'edop' / 'areas'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUT}')

RADIUS_M = RADIUS_KM * 1000
TABLE    = f'public.basin{LEVEL}'

Connected: ('cedop',)
Output dir: /Users/karlg/Documents/repos/_edops/output/edop/areas


In [3]:
# Cell 3 — rebuild Timbuktu buffer (step1 exact construction)
# Build reusable SQL fragment strings; geography buffer gives true-metre circle.
_PT_SQL   = f"ST_SetSRID(ST_MakePoint({LON}, {LAT}), 4326)"
_BUF_GEOG = f"ST_Buffer({_PT_SQL}::geography, {RADIUS_M})"
_BUF_GEOM = f"({_BUF_GEOG})::geometry"

buf_row = conn.execute(f"""
    SELECT
        ST_AsText({_BUF_GEOM}) AS buf_wkt,
        ST_Area({_BUF_GEOG})   AS buf_area_m2
""").fetchone()

BUF_WKT     = buf_row[0]
BUF_AREA_M2 = float(buf_row[1])

buf_geom = shapely_wkt.loads(BUF_WKT)
buf_gdf  = gpd.GeoDataFrame(geometry=[buf_geom], crs='EPSG:4326')

print(f'Buffer area : {BUF_AREA_M2/1e6:.1f} km²')
print(f'WKT length  : {len(BUF_WKT)} chars')
print('Buffer matches step1 construction: geography-type circle, radius in true metres')

Buffer area : 31237.5 km²
WKT length  : 1254 chars
Buffer matches step1 construction: geography-type circle, radius in true metres


In [4]:
# Cell 4 — Step 7.1a: HYDE grid reconnaissance
# Find all hyde_cells intersecting the buffer; compute overlap area per cell.
# ECC (effective-cell-count) Herfindahl form: 1/Σwᵢ²,  wᵢ = overlap_i / total_covered
# Absent cells (ocean, uncharted land) simply do not appear in the table — no zero rows.

hyde_recon_sql = f"""
WITH buf AS (
    SELECT {_BUF_GEOM} AS buf_geom
),
cells AS (
    SELECT
        hc.area_km2,
        ST_Area(ST_Intersection(hc.geom, buf.buf_geom)::geography) AS overlap_m2
    FROM temporal.hyde_cells hc, buf
    WHERE ST_Intersects(hc.geom, buf.buf_geom)
)
SELECT area_km2, overlap_m2
FROM cells
WHERE overlap_m2 > 0
ORDER BY overlap_m2 DESC
"""

print('Querying HYDE cells (2.2M rows, may take ~10–30 s)...')
hyde_raw = pd.read_sql(hyde_recon_sql, conn)
print(f'  {len(hyde_raw)} cells returned')

total_covered_hyde = hyde_raw['overlap_m2'].sum()
w_h = hyde_raw['overlap_m2'] / total_covered_hyde
HYDE_ECC      = float(1.0 / (w_h ** 2).sum())
HYDE_COVERAGE = total_covered_hyde / BUF_AREA_M2

print(f'\nHYDE reconnaissance')
print(f'  Cells intersecting buffer : {len(hyde_raw)}')
print(f'  Covered area              : {total_covered_hyde/1e6:.1f} km²  (buffer {BUF_AREA_M2/1e6:.1f} km²)')
print(f'  Coverage fraction         : {HYDE_COVERAGE:.4f}')
print(f'  Overlap range             : {hyde_raw["overlap_m2"].min()/1e6:.3f} – {hyde_raw["overlap_m2"].max()/1e6:.3f} km²')
print(f'  ECC                       : {HYDE_ECC:.1f}')
print(f'  Cell area range (km²)     : {hyde_raw["area_km2"].min():.1f} – {hyde_raw["area_km2"].max():.1f}')

Querying HYDE cells (2.2M rows, may take ~10–30 s)...
  426 cells returned

HYDE reconnaissance
  Cells intersecting buffer : 426
  Covered area              : 31237.5 km²  (buffer 31237.5 km²)
  Coverage fraction         : 1.0000
  Overlap range             : 0.033 – 82.235 km²
  ECC                       : 392.9
  Cell area range (km²)     : 81.5 – 82.3


In [5]:
# Cell 5 — Step 7.1b: LMR grid reconnaissance
# Registration confirmed from precompute_lmr_notches.py: stored lat/lon are cell centres;
# footprint = ±1° box.  lon stored 0–360; convert to −180/180 before footprint construction.
# Gapless tiling is guaranteed by the regular 2° grid; we report the spacing to confirm.

lmr_recon_sql = f"""
WITH buf AS (
    SELECT {_BUF_GEOM} AS buf_geom
),
footprints AS (
    SELECT
        lc.lat,
        CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END AS lon_disp,
        ST_MakeEnvelope(
            CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END - 1,
            lc.lat - 1,
            CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END + 1,
            lc.lat + 1,
            4326
        ) AS fp
    FROM temporal.lmr_climate lc
)
SELECT
    f.lat,
    f.lon_disp,
    ST_Area(f.fp::geography)                                        AS footprint_area_m2,
    ST_Area(ST_Intersection(f.fp, buf.buf_geom)::geography)         AS overlap_m2
FROM footprints f, buf
WHERE ST_Intersects(f.fp, buf.buf_geom)
ORDER BY overlap_m2 DESC
"""

print('Querying LMR footprints (16,380 rows)...')
lmr_raw = pd.read_sql(lmr_recon_sql, conn)
print(f'  {len(lmr_raw)} footprint(s) returned')

total_covered_lmr = lmr_raw['overlap_m2'].sum()
w_l = lmr_raw['overlap_m2'] / total_covered_lmr
LMR_ECC      = float(1.0 / (w_l ** 2).sum()) if len(lmr_raw) > 1 else 1.0
LMR_COVERAGE = total_covered_lmr / BUF_AREA_M2

print(f'\nLMR reconnaissance')
print(f'  Footprint(s) touching buffer : {len(lmr_raw)}')
print(f'  Covered area                 : {total_covered_lmr/1e6:.1f} km²  (buffer {BUF_AREA_M2/1e6:.1f} km²)')
print(f'  Coverage fraction            : {LMR_COVERAGE:.4f}')
print(f'  ECC                          : {LMR_ECC:.4f}')
print()

lmr_disp = lmr_raw.copy()
lmr_disp['footprint_km2'] = lmr_disp['footprint_area_m2'] / 1e6
lmr_disp['overlap_km2']   = lmr_disp['overlap_m2'] / 1e6
lmr_disp['pct_of_cell']   = lmr_disp['overlap_m2'] / lmr_disp['footprint_area_m2'] * 100
print('Intersecting LMR cell(s):')
print(lmr_disp[['lat', 'lon_disp', 'footprint_km2', 'overlap_km2', 'pct_of_cell']].to_string(
    index=False, float_format=lambda x: f'{x:.2f}'))

if len(lmr_raw) > 1:
    lat_spacings = np.diff(sorted(lmr_raw['lat'].unique()))
    lon_spacings = np.diff(sorted(lmr_raw['lon_disp'].unique()))
    print(f'\nGrid spacing: lat Δ={set(lat_spacings.round(1))}  lon Δ={set(lon_spacings.round(1))}')
    print('  (both should be {2.0} for a regular 2° grid)')
else:
    print('\nSingle cell — tiling regularity confirmed by regular 2° grid construction')

Querying LMR footprints (16,380 rows)...
  4 footprint(s) returned

LMR reconnaissance
  Footprint(s) touching buffer : 4
  Covered area                 : 31237.5 km²  (buffer 31237.5 km²)
  Coverage fraction            : 1.0000
  ECC                          : 3.7472

Intersecting LMR cell(s):
  lat  lon_disp  footprint_km2  overlap_km2  pct_of_cell
16.00     -2.00       47378.52     10044.08        21.20
16.00     -4.00       47378.52      9614.89        20.29
18.00     -2.00       46887.50      5930.46        12.65
18.00     -4.00       46887.50      5648.10        12.05

Grid spacing: lat Δ={np.float64(2.0)}  lon Δ={np.float64(2.0)}
  (both should be {2.0} for a regular 2° grid)


In [6]:
# Cell 6 — Step 7.1: reconnaissance summary
print('=' * 66)
print('Step 7.1 — Grid reconnaissance  |  Timbuktu 100 km / L06')
print('=' * 66)
print(f'{"Dataset":<10} {"Cells":>7} {"ECC":>8} {"Coverage":>10}  Routing')
print('-' * 66)
print(f'{"HYDE":<10} {len(hyde_raw):>7} {HYDE_ECC:>8.1f} {HYDE_COVERAGE:>9.4f}   '
      f'distribution  (ECC {HYDE_ECC:.1f} >> {ECC_THRESHOLD})')
print(f'{"LMR":<10} {len(lmr_raw):>7} {LMR_ECC:>8.3f} {LMR_COVERAGE:>9.4f}   '
      f'collapse      (ECC {LMR_ECC:.3f} ≤ {ECC_THRESHOLD})')
print('=' * 66)
print()
print(f'Provisional ECC_THRESHOLD = {ECC_THRESHOLD}')
print(f'  HYDE ({HYDE_ECC:.1f}) > {ECC_THRESHOLD}  → report distribution')
print(f'  LMR  ({LMR_ECC:.3f}) ≤ {ECC_THRESHOLD}  → collapse to area-weighted mean')
print()
print('Routing premise validated. Proceed to step 7.2 (diagnostic figure).')

Step 7.1 — Grid reconnaissance  |  Timbuktu 100 km / L06
Dataset      Cells      ECC   Coverage  Routing
------------------------------------------------------------------
HYDE           426    392.9    1.0000   distribution  (ECC 392.9 >> 10)
LMR              4    3.747    1.0000   collapse      (ECC 3.747 ≤ 10)

Provisional ECC_THRESHOLD = 10
  HYDE (392.9) > 10  → report distribution
  LMR  (3.747) ≤ 10  → collapse to area-weighted mean

Routing premise validated. Proceed to step 7.2 (diagnostic figure).


In [7]:
# Cell 7 — Step 7.2: fetch geodataframes for diagnostic figure
# HYDE: cells with geometry + overlap fraction of each cell
hyde_fig_sql = f"""
WITH buf AS (SELECT {_BUF_GEOM} AS buf_geom),
cells AS (
    SELECT
        hc.geom,
        hc.area_km2,
        ST_Area(ST_Intersection(hc.geom, buf.buf_geom)::geography) AS overlap_m2
    FROM temporal.hyde_cells hc, buf
    WHERE ST_Intersects(hc.geom, buf.buf_geom)
)
SELECT geom, area_km2, overlap_m2,
       overlap_m2 / (area_km2 * 1e6) AS overlap_frac
FROM cells
WHERE overlap_m2 > 0
"""
print('Fetching HYDE cell geometries...')
hyde_gdf = gpd.read_postgis(hyde_fig_sql, conn, geom_col='geom').rename_geometry('geometry')
print(f'  {len(hyde_gdf)} cells')

# LMR: footprint polygons (reconstructed from centre + ±1° box) + overlap fraction of each cell
lmr_fig_sql = f"""
WITH buf AS (SELECT {_BUF_GEOM} AS buf_geom),
footprints AS (
    SELECT
        lc.lat,
        CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END AS lon_disp,
        ST_MakeEnvelope(
            CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END - 1,
            lc.lat - 1,
            CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END + 1,
            lc.lat + 1, 4326) AS geom
    FROM temporal.lmr_climate lc
)
SELECT f.lat, f.lon_disp, f.geom,
       ST_Area(f.geom::geography)                                AS footprint_area_m2,
       ST_Area(ST_Intersection(f.geom, buf.buf_geom)::geography) AS overlap_m2
FROM footprints f, buf
WHERE ST_Intersects(f.geom, buf.buf_geom)
ORDER BY overlap_m2 DESC
"""
print('Fetching LMR footprint geometries...')
lmr_gdf = gpd.read_postgis(lmr_fig_sql, conn, geom_col='geom').rename_geometry('geometry')
lmr_gdf['overlap_frac'] = lmr_gdf['overlap_m2'] / lmr_gdf['footprint_area_m2']
print(f'  {len(lmr_gdf)} footprints')

Fetching HYDE cell geometries...
  426 cells
Fetching LMR footprint geometries...
  4 footprints


In [8]:
# Cell 8 — Step 7.2: diagnostic figure (resolution contrast)
# Left panel : HYDE dense swarm (426 cells, ECC 393) → report distribution
# Right panel: buffer straddling 4 LMR 2° cells (ECC 3.7) → collapse to weighted mean
# Style follows step1_buffer_resolver.ipynb.

BG_COLOR  = '#e8eef4'
BUF_COLOR = '#e63946'

# Extents: left zooms to buffer + margin; right zooms to the 4 LMR cells + margin
buf_b = buf_gdf.total_bounds          # [minx, miny, maxx, maxy]
mh    = (buf_b[2] - buf_b[0]) * 0.12
left_xlim = (buf_b[0] - mh, buf_b[2] + mh)
left_ylim = (buf_b[1] - mh, buf_b[3] + mh)

lmr_b = lmr_gdf.total_bounds
ml    = (lmr_b[2] - lmr_b[0]) * 0.10
right_xlim = (lmr_b[0] - ml, lmr_b[2] + ml)
right_ylim = (lmr_b[1] - ml, lmr_b[3] + ml)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('white')

# ---- Left panel: HYDE ----
ax1.set_facecolor(BG_COLOR)
norm_h = Normalize(vmin=0, vmax=1)
hyde_gdf.plot(column='overlap_frac', ax=ax1, cmap='Blues', norm=norm_h,
              edgecolor='#555', linewidth=0.3, alpha=0.85)
buf_gdf.boundary.plot(ax=ax1, color=BUF_COLOR, linewidth=2, linestyle='--',
                      label='Buffer (100 km)')
ax1.plot(LON, LAT, marker='*', color=BUF_COLOR, markersize=16, zorder=5,
         label='Query point')
ax1.set_xlim(*left_xlim)
ax1.set_ylim(*left_ylim)
ax1.set_title(
    f'HYDE 3.4   (~5 arcmin  ≈  9 km cells)\n'
    f'{len(hyde_gdf)} cells  ·  ECC = {HYDE_ECC:.1f}  ·  coverage = {HYDE_COVERAGE:.4f}\n'
    f'→  report area-weighted distribution',
    fontsize=11, color='black', pad=8)
ax1.set_axis_off()
leg1 = ax1.legend(fontsize=10, loc='lower right', frameon=True)
for t in leg1.get_texts():
    t.set_color('black')
sm1 = plt.cm.ScalarMappable(cmap='Blues', norm=norm_h)
sm1.set_array([])
cb1 = fig.colorbar(sm1, ax=ax1, fraction=0.03, pad=0.02, shrink=0.60)
cb1.set_label('Fraction of cell inside buffer', fontsize=10, color='black')
cb1.ax.yaxis.set_tick_params(color='black')
plt.setp(cb1.ax.yaxis.get_ticklabels(), color='black')

# ---- Right panel: LMR ----
ax2.set_facecolor(BG_COLOR)
norm_l = Normalize(vmin=0, vmax=lmr_gdf['overlap_frac'].max() * 1.2)
lmr_gdf.plot(column='overlap_frac', ax=ax2, cmap='Blues', norm=norm_l,
              edgecolor='#333', linewidth=1.5, alpha=0.75)
buf_gdf.boundary.plot(ax=ax2, color=BUF_COLOR, linewidth=2.5, linestyle='--',
                      label='Buffer (100 km)')
ax2.plot(LON, LAT, marker='*', color=BUF_COLOR, markersize=16, zorder=5,
         label='Query point')

# Label each LMR cell with its overlap fraction
for _, row in lmr_gdf.iterrows():
    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y
    ax2.text(cx, cy,
             f"({row['lat']:.0f}°, {row['lon_disp']:.0f}°)\n{row['overlap_frac']*100:.1f}% of cell",
             fontsize=9, ha='center', va='center', color='black', fontweight='bold')

ax2.set_xlim(*right_xlim)
ax2.set_ylim(*right_ylim)
ax2.set_title(
    f'LMR v2.1   (2°×2°  ≈  222 km cells at this latitude)\n'
    f'{len(lmr_gdf)} cells  ·  ECC = {LMR_ECC:.2f}  ·  coverage = {LMR_COVERAGE:.4f}\n'
    f'→  collapse to area-weighted mean  (LMR caveat applies)',
    fontsize=11, color='black', pad=8)
ax2.set_axis_off()
leg2 = ax2.legend(fontsize=10, loc='lower right', frameon=True)
for t in leg2.get_texts():
    t.set_color('black')
sm2 = plt.cm.ScalarMappable(cmap='Blues', norm=norm_l)
sm2.set_array([])
cb2 = fig.colorbar(sm2, ax=ax2, fraction=0.03, pad=0.02, shrink=0.60)
cb2.set_label('Fraction of cell inside buffer', fontsize=10, color='black')
cb2.ax.yaxis.set_tick_params(color='black')
plt.setp(cb2.ax.yaxis.get_ticklabels(), color='black')

fig.suptitle(
    'Step 7.2 — Grid resolution contrast  |  Timbuktu 100 km / L06\n'
    'HYDE and LMR route oppositely not because they are hardcoded,\n'
    'but because their cells differ by ~3 orders of magnitude in area  (ECC 393 vs 3.7)',
    fontsize=11, color='black', y=1.01)

outpath = OUT / 'step3b_block7_resolution_contrast.png'
fig.savefig(outpath, dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig)
display(IPImage(str(outpath)))
print(f'Saved: {outpath}')

Saved: /Users/karlg/Documents/repos/_edops/output/edop/areas/step3b_block7_resolution_contrast.png


In [9]:
# Cell 9 — Step 7.3: confirm diagnostic routing; record provisional ECC threshold
#
# ECC_THRESHOLD separates "collapse to weighted mean" (too few cells for a meaningful
# distribution) from "report distribution" (enough cells).  One fixture only; provisional.

print("Step 7.3 — Diagnostic threshold  |  Timbuktu 100 km / L06")
print()
print("ECC values from step 7.1 reconnaissance:")
print(f"  HYDE  {HYDE_ECC:6.1f}   (426 cells,  ~9 km each,  ~81 km² each)")
print(f"  LMR   {LMR_ECC:6.3f}   (  4 cells, ~222 km each, ~47 000 km² each)")
print()

print(f"Provisional  ECC_THRESHOLD = {ECC_THRESHOLD}")
print()

for name, ecc, expected in [("HYDE", HYDE_ECC, "distribution"), ("LMR", LMR_ECC, "collapse")]:
    routing = "distribution" if ecc > ECC_THRESHOLD else "collapse"
    mark    = "✓" if routing == expected else "✗"
    rel     = ">" if ecc > ECC_THRESHOLD else "≤"
    print(f"  {name:<6}  ECC = {ecc:7.2f}  {rel} {ECC_THRESHOLD}  →  {routing:<14}  {mark}")

print()
print(f"ECC_THRESHOLD = {ECC_THRESHOLD} cleanly separates the two datasets.")
print(f"Headroom below threshold: {ECC_THRESHOLD / LMR_ECC:.1f}×  (LMR ECC = {LMR_ECC:.2f})")
print(f"Headroom above threshold: {HYDE_ECC / ECC_THRESHOLD:.0f}×  (HYDE ECC = {HYDE_ECC:.1f})")
print()
print("Provisional — calibrated on a single fixture, single buffer radius, L06 only.")
print("Register row added; points at multi-fixture calibration step (alongside T=20,")
print("MODALITY_GAP, MIN_REGIME_WEIGHT, ZERO_FRACTION_THRESHOLD).")

Step 7.3 — Diagnostic threshold  |  Timbuktu 100 km / L06

ECC values from step 7.1 reconnaissance:
  HYDE   392.9   (426 cells,  ~9 km each,  ~81 km² each)
  LMR    3.747   (  4 cells, ~222 km each, ~47 000 km² each)

Provisional  ECC_THRESHOLD = 10

  HYDE    ECC =  392.94  > 10  →  distribution    ✓
  LMR     ECC =    3.75  ≤ 10  →  collapse        ✓

ECC_THRESHOLD = 10 cleanly separates the two datasets.
Headroom below threshold: 2.7×  (LMR ECC = 3.75)
Headroom above threshold: 39×  (HYDE ECC = 392.9)

Provisional — calibrated on a single fixture, single buffer radius, L06 only.
Register row added; points at multi-fixture calibration step (alongside T=20,
MODALITY_GAP, MIN_REGIME_WEIGHT, ZERO_FRACTION_THRESHOLD).


In [10]:
# Cell 10 — Step 7.4a: resolve time indices; fetch raw data for year 1150

# --- HYDE epoch: largest year_ce ≤ target (epoch applies until the next one starts) ---
epoch_row = conn.execute(f"""
    SELECT step_idx, year_ce
    FROM temporal.hyde_times
    WHERE year_ce <= {SINGLE_YEAR}
    ORDER BY year_ce DESC
    LIMIT 1
""").fetchone()
HYDE_STEP_IDX   = int(epoch_row[0])
HYDE_EPOCH_YEAR = int(epoch_row[1])
# PostgreSQL arrays are 1-indexed; step_idx is 0-indexed → access at [step_idx + 1]
HYDE_PG_IDX = HYDE_STEP_IDX + 1
print(f"HYDE epoch for year {SINGLE_YEAR}: step_idx={HYDE_STEP_IDX}  year_ce={HYDE_EPOCH_YEAR}  pg_idx={HYDE_PG_IDX}")

# --- LMR array index: year Y CE → arr[Y+1] (confirmed from temporal.py) ---
LMR_PG_IDX = SINGLE_YEAR + 1
print(f"LMR array index for year {SINGLE_YEAR}: {LMR_PG_IDX}")

# --- HYDE cell values at epoch ---
hyde_vals_sql = f"""
WITH buf AS (SELECT {_BUF_GEOM} AS buf_geom),
cells AS (
    SELECT
        hc.area_km2,
        ST_Area(ST_Intersection(hc.geom, buf.buf_geom)::geography) AS overlap_m2,
        hc.cropland [{HYDE_PG_IDX}]  AS cropland,
        hc.grazing  [{HYDE_PG_IDX}]  AS grazing,
        hc.pasture  [{HYDE_PG_IDX}]  AS pasture,
        hc.rangeland[{HYDE_PG_IDX}]  AS rangeland
    FROM temporal.hyde_cells hc, buf
    WHERE ST_Intersects(hc.geom, buf.buf_geom)
)
SELECT area_km2, overlap_m2, cropland, grazing, pasture, rangeland
FROM cells
WHERE overlap_m2 > 0
ORDER BY overlap_m2 DESC
"""
print(f"\nFetching HYDE cell values (epoch {HYDE_EPOCH_YEAR}, may take ~10–30 s)...")
hyde_vals = pd.read_sql(hyde_vals_sql, conn)
print(f"  {len(hyde_vals)} cells returned")
for v in ('cropland', 'grazing', 'pasture', 'rangeland'):
    print(f"  {v:<12}: {hyde_vals[v].min():.4f} – {hyde_vals[v].max():.4f} km²  "
          f"(nonzero: {(hyde_vals[v] > 0).sum()})")

# --- LMR values at year 1150 ---
lmr_vals_sql = f"""
WITH buf AS (SELECT {_BUF_GEOM} AS buf_geom),
footprints AS (
    SELECT
        lc.lat,
        CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END AS lon_disp,
        lc.pdsi [{LMR_PG_IDX}]  AS pdsi,
        lc.air  [{LMR_PG_IDX}]  AS air,
        lc.prate[{LMR_PG_IDX}]  AS prate,
        ST_MakeEnvelope(
            CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END - 1,
            lc.lat - 1,
            CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END + 1,
            lc.lat + 1, 4326) AS fp
    FROM temporal.lmr_climate lc
)
SELECT f.lat, f.lon_disp, f.pdsi, f.air, f.prate,
       ST_Area(f.fp::geography)                                        AS footprint_area_m2,
       ST_Area(ST_Intersection(f.fp, buf.buf_geom)::geography)         AS overlap_m2
FROM footprints f, buf
WHERE ST_Intersects(f.fp, buf.buf_geom)
ORDER BY overlap_m2 DESC
"""
print(f"\nFetching LMR values for year {SINGLE_YEAR}...")
lmr_vals = pd.read_sql(lmr_vals_sql, conn)
print(f"  {len(lmr_vals)} footprints")
print(lmr_vals[['lat','lon_disp','pdsi','air','prate','overlap_m2']].to_string(index=False,
    float_format=lambda x: f'{x:.5f}'))

# --- eVolv2k events at year 1150 ---
evolv_sql = f"""
SELECT year_ad, vssi_tg, so4_grl, so4_ant, lat, location
FROM temporal.evolv2k_v4
WHERE year_ad = {SINGLE_YEAR}
ORDER BY vssi_tg DESC
"""
evolv_events = pd.read_sql(evolv_sql, conn)
print(f"\neVolv2k events at year {SINGLE_YEAR}: {len(evolv_events)}"
      + ("" if len(evolv_events) == 0 else "\n" + evolv_events.to_string(index=False)))

HYDE epoch for year 1150: step_idx=21  year_ce=1100  pg_idx=22
LMR array index for year 1150: 1151

Fetching HYDE cell values (epoch 1100, may take ~10–30 s)...
  426 cells returned
  cropland    : 0.0000 – 0.0989 km²  (nonzero: 410)
  grazing     : 0.0394 – 32.4281 km²  (nonzero: 426)
  pasture     : 0.0000 – 0.0000 km²  (nonzero: 0)
  rangeland   : 0.0394 – 32.4281 km²  (nonzero: 426)

Fetching LMR values for year 1150...
  4 footprints
     lat  lon_disp    pdsi      air   prate        overlap_m2
16.00000  -2.00000 0.27683 -0.39364 0.00000 10044084183.19399
16.00000  -4.00000 0.23465 -0.38830 0.00000  9614887335.87552
18.00000  -2.00000 0.12855 -0.35884 0.00000  5930462689.51404
18.00000  -4.00000 0.22326 -0.33956 0.00000  5648100688.42211

eVolv2k events at year 1150: 0


In [11]:
# Cell 11 — Step 7.4b: compute weighted aggregates; assemble shared envelope
#
# Weights = overlap area normalized over data-bearing cells.
# Zeros are real data (e.g., desert cropland = 0); only NULLs are excluded.
# prate: raw DB value is kg/m²/s; × 86400 → mm/day  (matches temporal.py).

LMR_CAVEAT = (
    "Anomaly vs. 850–1850 CE CCSM4 reference frame; "
    "spatial structure reflects reanalysis prior. "
    "At buffer scale the collapsed value is essentially the prior at this location."
)

def _weighted_quantile(vals, weights, q):
    """Weighted quantile via sorted cumulative weights + linear interpolation."""
    idx  = np.argsort(vals)
    sv   = vals[idx];  sw = weights[idx]
    cumw = np.cumsum(sw) / np.sum(sw)
    return float(np.interp(q, cumw, sv))

def _agg_hyde(df, var_col):
    """Area-weighted mean + p10/p90/sd for one HYDE variable (distribution path)."""
    valid = df[df[var_col].notna()].copy()
    if len(valid) == 0:
        return dict(representative_raw=None, p10=None, p90=None, sd=None,
                    n_units=0, coverage_weight=0.0, status='no_data')
    tot_w = float(valid['overlap_m2'].sum())
    w     = (valid['overlap_m2'] / tot_w).values
    v     = valid[var_col].values.astype(float)
    mean_ = float(np.dot(v, w))
    return dict(
        representative_raw = mean_,
        p10                = _weighted_quantile(v, w, 0.1),
        p90                = _weighted_quantile(v, w, 0.9),
        sd                 = float(np.sqrt(np.dot(w, (v - mean_) ** 2))),
        n_units            = len(valid),
        coverage_weight    = tot_w / BUF_AREA_M2,
        status             = 'ok',
    )

def _agg_lmr(df, var_col, scale=1.0):
    """Area-weighted mean for one LMR variable (collapse path; distribution suppressed)."""
    valid = df[df[var_col].notna()].copy()
    if len(valid) == 0:
        return dict(representative_raw=None, n_units=0, coverage_weight=0.0, status='no_data')
    tot_w = float(valid['overlap_m2'].sum())
    w     = (valid['overlap_m2'] / tot_w).values
    v     = valid[var_col].values.astype(float) * scale
    return dict(
        representative_raw = float(np.dot(v, w)),
        n_units            = len(valid),
        coverage_weight    = tot_w / BUF_AREA_M2,
        status             = 'ok',
    )

def _row(variable, method, agg, units, unit_type, year,
         epoch_year=None, lmr_caveat=None):
    """Build one shared-envelope row.  p10/p90/sd come from agg if present (None otherwise)."""
    return dict(
        variable             = variable,
        method               = method,
        status               = agg['status'],
        representative_score = None,
        representative_raw   = agg.get('representative_raw'),
        units                = units,
        n_units              = agg.get('n_units', 0),
        unit_type            = unit_type,
        coverage_weight      = agg.get('coverage_weight'),
        year                 = year,
        epoch_year           = epoch_year,
        p10                  = agg.get('p10'),
        p90                  = agg.get('p90'),
        sd                   = agg.get('sd'),
        lmr_caveat           = lmr_caveat,
    )

rows = []

# HYDE: 4 vars, grid_areal_distribution
for var, units in [('cropland','km²'), ('grazing','km²'), ('pasture','km²'), ('rangeland','km²')]:
    rows.append(_row(f'hyde_{var}', 'grid_areal_distribution',
                     _agg_hyde(hyde_vals, var), units,
                     'hyde_cell', SINGLE_YEAR, epoch_year=HYDE_EPOCH_YEAR))

# LMR: 3 vars, grid_areal_collapsed; prate × 86400 → mm/day
for api_name, col, scale, units in [
    ('pdsi',  'pdsi',  1.0,     'dimensionless anomaly'),
    ('air',   'air',   1.0,     'K anomaly'),
    ('prate', 'prate', 86400.0, 'mm/day anomaly'),
]:
    rows.append(_row(f'lmr_{api_name}', 'grid_areal_collapsed',
                     _agg_lmr(lmr_vals, col, scale=scale), units,
                     'lmr_cell', SINGLE_YEAR, lmr_caveat=LMR_CAVEAT))

# eVolv2k: global forcing, no spatial step
if len(evolv_events) == 0:
    rows.append(_row('evolv2k_vssi', 'global_forcing',
                     dict(representative_raw=None, n_units=0,
                          coverage_weight=None, status='no_events'),
                     'Tg S', 'global', SINGLE_YEAR))
else:
    for _, ev in evolv_events.iterrows():
        rows.append(_row('evolv2k_vssi', 'global_forcing',
                         dict(representative_raw=float(ev['vssi_tg']), n_units=1,
                              coverage_weight=None, status='ok'),
                         'Tg S', 'global', int(ev['year_ad'])))

envelope_74 = pd.DataFrame(rows)
print(f"Envelope assembled: {len(envelope_74)} rows")

Envelope assembled: 8 rows


In [12]:
# Cell 12 — Step 7.4c: display shared envelope (single time step, year 1150)

pd.set_option('display.float_format', lambda x: f'{x:.5f}')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

# Core envelope: one row per variable
core = ['variable', 'method', 'status', 'representative_raw', 'units',
        'n_units', 'unit_type', 'coverage_weight', 'year', 'epoch_year']
print("Core envelope:")
print(envelope_74[core].to_string(index=False))
print()

# Distribution detail (HYDE rows only)
hyde_env = envelope_74[envelope_74['method'] == 'grid_areal_distribution']
print("HYDE distribution detail (area-weighted):")
print(hyde_env[['variable', 'representative_raw', 'p10', 'p90', 'sd']].to_string(index=False))
print()

# Sanity: HYDE spread at Timbuktu ~1150 CE
for _, r in hyde_env.iterrows():
    spread = (r['p90'] or 0) - (r['p10'] or 0)
    print(f"  {r['variable']:<18}  mean={r['representative_raw']:.4f}  "
          f"p10={r['p10']:.4f}  p90={r['p90']:.4f}  spread={spread:.4f}  km²")

Core envelope:
      variable                  method    status  representative_raw                 units  n_units unit_type  coverage_weight  year  epoch_year
 hyde_cropland grid_areal_distribution        ok             0.01678                   km²      426 hyde_cell          1.00000  1150  1100.00000
  hyde_grazing grid_areal_distribution        ok             6.26666                   km²      426 hyde_cell          1.00000  1150  1100.00000
  hyde_pasture grid_areal_distribution        ok             0.00000                   km²      426 hyde_cell          1.00000  1150  1100.00000
hyde_rangeland grid_areal_distribution        ok             6.26666                   km²      426 hyde_cell          1.00000  1150  1100.00000
      lmr_pdsi    grid_areal_collapsed        ok             0.22601 dimensionless anomaly        4  lmr_cell          1.00000  1150         NaN
       lmr_air    grid_areal_collapsed        ok            -0.37561             K anomaly        4  lmr_cell      

In [13]:
# Cell 13 — Step 7.5: aggregate_band_t — spatial collapse over a time span
#
# Same function handles snapshot (narrow span → few steps) and history (wide span → many).
# No mode flag: span width alone determines the output.
#
# HYDE: spatial intersection done ONCE via CROSS JOIN with epoch list — avoids N round-trips.
# LMR:  array slices fetched once; expanded year-by-year in Python.
# eVolv2k: single range query; global forcing, no spatial step.

LMR_RANGE   = (0,    1998)   # LMR v2.1 coverage
EVOLV_RANGE = (-491, 1890)   # eVolv2k v4 catalog range

def aggregate_band_t(from_year, to_year):
    rows = []

    # ── HYDE ──────────────────────────────────────────────────────────────────
    # The cell_overlaps CTE runs the spatial filter once; CROSS JOIN with epochs
    # extracts the right array element for each epoch without re-scanning hyde_cells.
    hyde_sql = f"""
    WITH buf AS (SELECT {_BUF_GEOM} AS buf_geom),
    epochs AS (
        SELECT step_idx, year_ce
        FROM temporal.hyde_times
        WHERE year_ce BETWEEN {from_year} AND {to_year}
    ),
    cell_overlaps AS (
        SELECT area_km2,
               ST_Area(ST_Intersection(hc.geom, buf.buf_geom)::geography) AS overlap_m2,
               cropland, grazing, pasture, rangeland
        FROM temporal.hyde_cells hc, buf
        WHERE ST_Intersects(hc.geom, buf.buf_geom)
    )
    SELECT
        e.year_ce, e.step_idx,
        co.area_km2, co.overlap_m2,
        co.cropland [e.step_idx + 1] AS cropland,
        co.grazing  [e.step_idx + 1] AS grazing,
        co.pasture  [e.step_idx + 1] AS pasture,
        co.rangeland[e.step_idx + 1] AS rangeland
    FROM cell_overlaps co CROSS JOIN epochs e
    WHERE co.overlap_m2 > 0
    ORDER BY e.year_ce
    """
    hyde_ts = pd.read_sql(hyde_sql, conn)

    for year_ce, grp in hyde_ts.groupby('year_ce', sort=True):
        for var, units in [('cropland','km²'), ('grazing','km²'),
                           ('pasture','km²'),  ('rangeland','km²')]:
            rows.append(_row(f'hyde_{var}', 'grid_areal_distribution',
                             _agg_hyde(grp, var), units,
                             'hyde_cell', int(year_ce), epoch_year=int(year_ce)))

    # ── LMR ───────────────────────────────────────────────────────────────────
    # Array slice fetched once per footprint; expanded to annual rows in Python.
    lmr_from = max(LMR_RANGE[0], from_year)
    lmr_to   = min(LMR_RANGE[1], to_year)

    if lmr_from <= lmr_to:
        pg_from, pg_to = lmr_from + 1, lmr_to + 1
        lmr_sql = f"""
        WITH buf AS (SELECT {_BUF_GEOM} AS buf_geom),
        footprints AS (
            SELECT
                lc.lat,
                CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END AS lon_disp,
                lc.pdsi [{pg_from}:{pg_to}]  AS pdsi_slice,
                lc.air  [{pg_from}:{pg_to}]  AS air_slice,
                lc.prate[{pg_from}:{pg_to}]  AS prate_slice,
                ST_MakeEnvelope(
                    CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END - 1,
                    lc.lat - 1,
                    CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END + 1,
                    lc.lat + 1, 4326) AS fp
            FROM temporal.lmr_climate lc
        )
        SELECT f.pdsi_slice, f.air_slice, f.prate_slice,
               ST_Area(ST_Intersection(f.fp, buf.buf_geom)::geography) AS overlap_m2
        FROM footprints f, buf
        WHERE ST_Intersects(f.fp, buf.buf_geom)
        ORDER BY overlap_m2 DESC
        """
        lmr_ts = pd.read_sql(lmr_sql, conn)

        ov      = lmr_ts['overlap_m2'].values.astype(float)
        w_lmr   = ov / ov.sum()
        cov_lmr = float(ov.sum() / BUF_AREA_M2)
        n_fp    = len(lmr_ts)
        years_l = list(range(lmr_from, lmr_to + 1))

        # Each *_slice column is a Python list (psycopg3 decodes PG arrays automatically)
        pdsi_mat  = np.array(lmr_ts['pdsi_slice'].tolist(),  dtype=float)   # (n_fp, n_years)
        air_mat   = np.array(lmr_ts['air_slice'].tolist(),   dtype=float)
        prate_mat = np.array(lmr_ts['prate_slice'].tolist(), dtype=float) * 86400

        for i, year in enumerate(years_l):
            for api_name, mat, units in [
                ('pdsi',  pdsi_mat,  'dimensionless anomaly'),
                ('air',   air_mat,   'K anomaly'),
                ('prate', prate_mat, 'mm/day anomaly'),
            ]:
                mean_val = float(np.dot(mat[:, i], w_lmr))
                rows.append(_row(f'lmr_{api_name}', 'grid_areal_collapsed',
                                 dict(representative_raw=mean_val, n_units=n_fp,
                                      coverage_weight=cov_lmr, status='ok'),
                                 units, 'lmr_cell', year))

    # ── eVolv2k ───────────────────────────────────────────────────────────────
    ev_from = max(EVOLV_RANGE[0], from_year)
    ev_to   = min(EVOLV_RANGE[1], to_year)

    if ev_from <= ev_to:
        events = pd.read_sql(f"""
            SELECT year_ad, vssi_tg, so4_grl, so4_ant, lat, location
            FROM temporal.evolv2k_v4
            WHERE year_ad BETWEEN {ev_from} AND {ev_to}
            ORDER BY year_ad
        """, conn)
        for _, ev in events.iterrows():
            rows.append(_row('evolv2k_vssi', 'global_forcing',
                             dict(representative_raw=float(ev['vssi_tg']),
                                  n_units=1, coverage_weight=None, status='ok'),
                             'Tg S', 'global', int(ev['year_ad'])))

    return pd.DataFrame(rows) if rows else pd.DataFrame()

In [14]:
# Cell 14 — Step 7.5a: run primary span (1100–1200) and wide span (1000–2000)
# Same function, no mode flag — span width determines the output length.
import time

print(f"=== Primary span ({PRIMARY_FROM}–{PRIMARY_TO} CE) ===")
t0 = time.time()
ts_primary = aggregate_band_t(PRIMARY_FROM, PRIMARY_TO)
elapsed = time.time() - t0
print(f"  {len(ts_primary)} rows total  ({elapsed:.1f}s)")
for meth, label in [('grid_areal_distribution', 'HYDE'),
                     ('grid_areal_collapsed',    'LMR'),
                     ('global_forcing',          'eVolv2k')]:
    sub = ts_primary[ts_primary['method'] == meth]
    if len(sub):
        n_vars  = sub['variable'].nunique()
        n_steps = sub['year'].nunique()
        dist    = "w/ p10/p90/sd" if meth == 'grid_areal_distribution' else "scalar only"
        print(f"  {label:<8}: {n_vars} vars × {n_steps} steps = {len(sub)} rows  ({dist})")

print()
print(f"=== Wide span ({WIDE_FROM}–{WIDE_TO} CE) ===")
t0 = time.time()
ts_wide = aggregate_band_t(WIDE_FROM, WIDE_TO)
elapsed = time.time() - t0
print(f"  {len(ts_wide)} rows total  ({elapsed:.1f}s)")
for meth, label in [('grid_areal_distribution', 'HYDE'),
                     ('grid_areal_collapsed',    'LMR'),
                     ('global_forcing',          'eVolv2k')]:
    sub = ts_wide[ts_wide['method'] == meth]
    if len(sub):
        n_vars  = sub['variable'].nunique()
        n_steps = sub['year'].nunique()
        dist    = "w/ p10/p90/sd" if meth == 'grid_areal_distribution' else "scalar only"
        print(f"  {label:<8}: {n_vars} vars × {n_steps} steps = {len(sub)} rows  ({dist})")

  3427 rows total  (0.5s)
  HYDE    : 4 vars × 83 steps = 332 rows  (w/ p10/p90/sd)
  LMR     : 3 vars × 999 steps = 2997 rows  (scalar only)
  eVolv2k : 1 vars × 98 steps = 98 rows  (scalar only)


In [15]:
# Cell 15 — Step 7.5b: payload asymmetry + HYDE distribution evolution
# The work order asks us to "watch the HYDE distribution change shape across epochs."
# Grazing is the active signal at Timbuktu; cropland and rangeland also shown.

print("─" * 72)
print("PAYLOAD ASYMMETRY (wide span 1000–2000 CE)")
print("─" * 72)
for meth, label, note in [
    ('grid_areal_distribution', 'HYDE',    'few epochs, each carries p10/p90/sd'),
    ('grid_areal_collapsed',    'LMR',     'many years, each a single collapsed mean'),
    ('global_forcing',          'eVolv2k', 'event rows only — no data = silent gap'),
]:
    sub = ts_wide[ts_wide['method'] == meth]
    if len(sub):
        print(f"  {label:<8}: {sub['variable'].nunique()} vars × {sub['year'].nunique():4d} steps"
              f"  →  {len(sub):4d} rows  [{note}]")
print()

# ── HYDE grazing evolution across epochs (selected centuries + modern decadal) ──
print("─" * 72)
print("HYDE distribution change — grazing (km² per cell)  |  Timbuktu 100 km")
print("─" * 72)
graz = (ts_wide[ts_wide['variable'] == 'hyde_grazing']
        .sort_values('epoch_year').reset_index(drop=True))

# Show all centennial epochs (1000–1700), then decadal pivots and annual end
epoch_yrs = sorted(graz['epoch_year'].unique())
milestones = ([y for y in epoch_yrs if y <= 1700] +      # all centuries
              [y for y in epoch_yrs if 1700 < y <= 1950 and y % 50 == 0] +  # half-centurials
              [y for y in epoch_yrs if y > 1950])         # annual (last 50 of range)
milestones = sorted(set(milestones))

disp = graz[graz['epoch_year'].isin(milestones)].copy()
print(f"  {'epoch':>6}  {'mean':>8}  {'p10':>8}  {'p90':>8}  {'spread':>8}  {'status'}")
print(f"  {'':->6}  {'':->8}  {'':->8}  {'':->8}  {'':->8}")
for _, r in disp.iterrows():
    sp = (r['p90'] or 0) - (r['p10'] or 0)
    print(f"  {int(r['epoch_year']):>6}  {r['representative_raw']:>8.4f}  "
          f"{r['p10']:>8.4f}  {r['p90']:>8.4f}  {sp:>8.4f}  {r['status']}")

print()
# ── LMR snapshot: PDSI 5-year running means, primary span ──
print("─" * 72)
print(f"LMR pdsi (area-weighted mean)  |  {PRIMARY_FROM}–{PRIMARY_TO} CE  (101 annual values)")
print("─" * 72)
pdsi_p = (ts_primary[ts_primary['variable'] == 'lmr_pdsi']
          .sort_values('year')['representative_raw'].values)
if len(pdsi_p) > 0:
    print(f"  mean = {pdsi_p.mean():.4f}   min = {pdsi_p.min():.4f}   max = {pdsi_p.max():.4f}")
    rm5 = pd.Series(pdsi_p).rolling(5, center=True).mean().dropna().values
    print(f"  5-yr rolling mean range: {rm5.min():.4f} – {rm5.max():.4f}")

print()
# ── Top eVolv2k events in wide span ──
print("─" * 72)
print("Top eVolv2k events (1000–1890 CE, by vssi_tg)")
print("─" * 72)
ev_wide = ts_wide[ts_wide['method'] == 'global_forcing'].nlargest(10, 'representative_raw')
print(f"  {'year':>6}  {'vssi_tg (Tg S)':>14}")
for _, r in ev_wide.iterrows():
    print(f"  {int(r['year']):>6}  {r['representative_raw']:>14.2f}")

────────────────────────────────────────────────────────────────────────
PAYLOAD ASYMMETRY (wide span 1000–2000 CE)
────────────────────────────────────────────────────────────────────────
  HYDE    : 4 vars ×   83 steps  →   332 rows  [few epochs, each carries p10/p90/sd]
  LMR     : 3 vars ×  999 steps  →  2997 rows  [many years, each a single collapsed mean]
  eVolv2k : 1 vars ×   98 steps  →    98 rows  [event rows only — no data = silent gap]

────────────────────────────────────────────────────────────────────────
HYDE distribution change — grazing (km² per cell)  |  Timbuktu 100 km
────────────────────────────────────────────────────────────────────────
   epoch      mean       p10       p90    spread  status
  ------  --------  --------  --------  --------
    1000    5.8297    0.2254   24.2064   23.9810  ok
    1100    6.2667    0.2183   26.1032   25.8848  ok
    1200    6.7033    0.2079   28.0099   27.8020  ok
    1300    7.2654    0.1975   30.4543   30.2568  ok
    1400    8

In [16]:
# Cell 16 — Step 7.6a: write primary span envelope to TSV
#
# Columns follow the shared envelope (Blocks 1–6) plus Block 7 extensions:
#   n_units / unit_type  (replaces n_basins — engine-assembly item, see register)
#   year / epoch_year    (temporal axis)
#   p10 / p90 / sd       (HYDE distribution detail; null for LMR and eVolv2k)
#   lmr_caveat           (mandatory on every LMR row; null elsewhere)
#
# HYDE 1950 transition artifact: the 1950 epoch value (last centennial/decadal point)
# spikes then reverts immediately in 1951 — a known HYDE 3.4 cadence-change artifact,
# not a real land-use signal.  Noted here; surfaced to consumer as a caveat column below.

HYDE_1950_CAVEAT = (
    "HYDE 3.4 cadence-transition artifact: 1950 is the last centennial/decadal epoch "
    "before annual data begins; value may not reflect real land-use change."
)

# Tag the 1950 HYDE rows in ts_primary (none expected, but pattern established for ts_wide)
ts_primary_out = ts_primary.copy()
hyde_1950 = (ts_primary_out['method'] == 'grid_areal_distribution') & (ts_primary_out['epoch_year'] == 1950)
if hyde_1950.any():
    ts_primary_out.loc[hyde_1950, 'hyde_caveat'] = HYDE_1950_CAVEAT
else:
    ts_primary_out['hyde_caveat'] = None

# Column order
COL_ORDER = [
    'variable', 'method', 'status',
    'representative_score', 'representative_raw', 'units',
    'n_units', 'unit_type', 'coverage_weight',
    'year', 'epoch_year',
    'p10', 'p90', 'sd',
    'lmr_caveat', 'hyde_caveat',
]
ts_primary_out = ts_primary_out.reindex(columns=COL_ORDER)

out_path = OUT / 'step3b_block7_primary.tsv'
ts_primary_out.to_csv(out_path, sep='\t', index=False)

print(f"Written: {out_path}")
print(f"  Shape : {ts_primary_out.shape}")
print()
print("Row counts by method:")
print(ts_primary_out.groupby(['method', 'variable'])['year'].count().rename('n_steps').to_string())
print()
print("Sample — HYDE rows (first epoch):")
print(ts_primary_out[ts_primary_out['method'] == 'grid_areal_distribution']
      .head(4)[['variable','epoch_year','representative_raw','p10','p90','sd','coverage_weight']]
      .to_string(index=False, float_format=lambda x: f'{x:.5f}'))
print()
print("Sample — LMR rows (first 3 years):")
print(ts_primary_out[ts_primary_out['method'] == 'grid_areal_collapsed']
      .head(3)[['variable','year','representative_raw','units','n_units','coverage_weight']]
      .to_string(index=False, float_format=lambda x: f'{x:.5f}'))

Written: /Users/karlg/Documents/repos/_edops/output/edop/areas/step3b_block7_primary.tsv
  Shape : (321, 16)

Row counts by method:
method                   variable      
global_forcing           evolv2k_vssi       10
grid_areal_collapsed     lmr_air           101
                         lmr_pdsi          101
                         lmr_prate         101
grid_areal_distribution  hyde_cropland       2
                         hyde_grazing        2
                         hyde_pasture        2
                         hyde_rangeland      2

Sample — HYDE rows (first epoch):
      variable  epoch_year  representative_raw     p10      p90       sd  coverage_weight
 hyde_cropland  1100.00000             0.01678 0.00000  0.07231  0.02800          1.00000
  hyde_grazing  1100.00000             6.26666 0.21833 26.10316 10.05048          1.00000
  hyde_pasture  1100.00000             0.00000 0.00000  0.00000  0.00000          1.00000
hyde_rangeland  1100.00000             6.26666 0.21833 26

In [17]:
# Cell 17 — Step 7.6b: write wide span envelope to TSV
ts_wide_out = ts_wide.copy()

hyde_1950_mask = (
    (ts_wide_out['method'] == 'grid_areal_distribution') &
    (ts_wide_out['epoch_year'] == 1950)
)
ts_wide_out['hyde_caveat'] = None
ts_wide_out.loc[hyde_1950_mask, 'hyde_caveat'] = HYDE_1950_CAVEAT

ts_wide_out = ts_wide_out.reindex(columns=COL_ORDER)

out_path_wide = OUT / 'step3b_block7_wide.tsv'
ts_wide_out.to_csv(out_path_wide, sep='\t', index=False)

print(f"Written: {out_path_wide}")
print(f"  Shape : {ts_wide_out.shape}")
print()
print("Row counts by method:")
print(ts_wide_out.groupby('method')['variable'].count().rename('n_rows').to_string())
print()
n_caveated = hyde_1950_mask.sum()
print(f"HYDE 1950 artifact rows tagged: {n_caveated}  (should be 4 — one per variable)")
print()
print("Largest eVolv2k events in wide span:")
ev_check = ts_wide_out[ts_wide_out['method'] == 'global_forcing'].nlargest(5, 'representative_raw')
print(ev_check[['year','representative_raw']].to_string(index=False, float_format=lambda x: f'{x:.2f}'))

Written: /Users/karlg/Documents/repos/_edops/output/edop/areas/step3b_block7_wide.tsv
  Shape : (3427, 16)

Row counts by method:
method
global_forcing               98
grid_areal_collapsed       2997
grid_areal_distribution     332

HYDE 1950 artifact rows tagged: 4  (should be 4 — one per variable)

Largest eVolv2k events in wide span:
 year  representative_raw
 1257               59.42
 1458               32.98
 1815               28.08
 1230               23.78
 1783               20.81


In [19]:
# Cell 18 — Step 7.6c: HYDE distribution companion TSV
# Extracts distribution shape per variable per epoch.
# spread_native: p90 - p10 in km²/cell (native HYDE units).
# Named `spread_native` not `spread` to avoid collision with Blocks 1–6
# where `spread` is in percentile points (0–100 scale).
# Note: p10/p90/sd columns in this file are also native units (km²/cell),
# unlike the pp-scale p10/p90 in step3_results.tsv — same column names,
# different units. Engine assembly must reconcile (register item).

hyde_dist = (
    ts_wide_out[ts_wide_out['method'] == 'grid_areal_distribution']
    [['variable', 'epoch_year', 'representative_raw', 'p10', 'p90', 'sd', 'n_units', 'coverage_weight', 'hyde_caveat']]
    .copy()
)
hyde_dist['spread_native'] = hyde_dist['p90'] - hyde_dist['p10']   # km²/cell, not pp
hyde_dist = hyde_dist.sort_values(['variable', 'epoch_year']).reset_index(drop=True)

hyde_dist = hyde_dist[['variable', 'epoch_year', 'representative_raw',
                        'p10', 'p90', 'spread_native', 'sd',
                        'n_units', 'coverage_weight', 'hyde_caveat']]

out_path_dist = OUT / 'step3b_block7_hyde_distributions.tsv'
hyde_dist.to_csv(out_path_dist, sep='\t', index=False)

print(f"Written: {out_path_dist}")
print(f"  Shape : {hyde_dist.shape}  ({hyde_dist['variable'].nunique()} vars × {hyde_dist['epoch_year'].nunique()} epochs)")
print()

for var in ['hyde_grazing', 'hyde_cropland']:
    sub = hyde_dist[hyde_dist['variable'] == var]
    print(f"{var}  ({len(sub)} epochs):")
    display_rows = sub[sub['epoch_year'].isin(
        [y for y in sub['epoch_year'] if y <= 1700] +
        [sub['epoch_year'].max()]
    )]
    print(display_rows[['epoch_year','representative_raw','p10','p90','spread_native']]
          .to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print()

Written: /Users/karlg/Documents/repos/_edops/output/edop/areas/step3b_block7_hyde_distributions.tsv
  Shape : (332, 10)  (4 vars × 83 epochs)

hyde_grazing  (83 epochs):
 epoch_year  representative_raw    p10     p90  spread_native
  1000.0000              5.8297 0.2254 24.2064        23.9810
  1100.0000              6.2667 0.2183 26.1032        25.8848
  1200.0000              6.7033 0.2079 28.0099        27.8020
  1300.0000              7.2654 0.1975 30.4543        30.2568
  1400.0000              8.0724 0.1886 33.9431        33.7545
  1500.0000              8.7409 0.1707 36.8684        36.6977
  1600.0000             10.1976 0.1601 43.1469        42.9868
  1700.0000             11.6521 0.1383 49.4545        49.3161
  2000.0000             14.8464 0.0056 61.9357        61.9301

hyde_cropland  (83 epochs):
 epoch_year  representative_raw    p10    p90  spread_native
  1000.0000              0.0156 0.0000 0.0673         0.0673
  1100.0000              0.0168 0.0000 0.0723         0.072

In [20]:
# Cell 19 — AF.3 diagnostic: source vs. seam at HYDE 1950
#
# If the spike at 1950 is in the source data, arr[step_idx+1] for year 1950 will reproduce
# the high value we see in aggregate, and adjacent steps (1900, 1951) will be lower.
# If it's a seam misalignment, the wrong array element is being pulled for 1950.

# ── Step 1: show the hyde_times cadence around the seam ───────────────────────
seam_years = pd.read_sql("""
    SELECT step_idx, year_ce FROM temporal.hyde_times
    WHERE year_ce BETWEEN 1890 AND 1960
    ORDER BY year_ce
""", conn)
print("hyde_times cadence around the seam:")
print(seam_years.to_string(index=False))
print()

# ── Step 2: for the three highest-overlap Timbuktu buffer cells,
#    read grazing directly at the steps for 1900, 1950, 1951 ──────────────────
seam_sql = f"""
WITH buf AS (SELECT {_BUF_GEOM} AS buf_geom),
top_cells AS (
    SELECT
        hc.ctid::text AS cell_id,
        hc.area_km2,
        ST_Area(ST_Intersection(hc.geom, buf.buf_geom)::geography) AS overlap_m2,
        hc.grazing
    FROM temporal.hyde_cells hc, buf
    WHERE ST_Intersects(hc.geom, buf.buf_geom)
    ORDER BY overlap_m2 DESC
    LIMIT 3
),
steps AS (
    SELECT step_idx, year_ce FROM temporal.hyde_times
    WHERE year_ce IN (1900, 1950, 1951)
)
SELECT
    tc.cell_id,
    tc.area_km2,
    tc.overlap_m2 / 1e6 AS overlap_km2,
    s.year_ce,
    s.step_idx,
    s.step_idx + 1 AS pg_idx,
    tc.grazing[s.step_idx + 1] AS grazing_value
FROM top_cells tc CROSS JOIN steps s
ORDER BY tc.overlap_m2 DESC, s.year_ce
"""
seam_vals = pd.read_sql(seam_sql, conn)
print("Raw grazing values at the seam (3 highest-overlap cells):")
print(seam_vals.to_string(index=False, float_format=lambda x: f'{x:.5f}'))
print()

# ── Step 3: verdict ───────────────────────────────────────────────────────────
vals_1950 = seam_vals[seam_vals['year_ce'] == 1950]['grazing_value'].values
vals_1900 = seam_vals[seam_vals['year_ce'] == 1900]['grazing_value'].values
vals_1951 = seam_vals[seam_vals['year_ce'] == 1951]['grazing_value'].values

spike_in_1950 = all(v > v2 and v > v3 for v, v2, v3 in zip(vals_1950, vals_1900, vals_1951))
print("Verdict:")
if spike_in_1950:
    print("  1950 values are higher than both 1900 and 1951 in all 3 cells.")
    print("  → Spike is IN THE SOURCE DATA. hyde_caveat stands; no indexing fix needed.")
else:
    print("  1950 values do NOT consistently exceed 1900 and 1951.")
    print("  → Possible seam misalignment. Check step_idx ↔ year_ce mapping carefully.")

hyde_times cadence around the seam:
 step_idx  year_ce
       46     1890
       47     1900
       48     1910
       49     1920
       50     1930
       51     1940
       52     1950
       53     1951
       54     1952
       55     1953
       56     1954
       57     1955
       58     1956
       59     1957
       60     1958
       61     1959
       62     1960

Raw grazing values at the seam (3 highest-overlap cells):
   cell_id  area_km2  overlap_km2  year_ce  step_idx  pg_idx  grazing_value
(392915,4)  82.23525     82.23525     1900        47      48       65.63199
(392915,3)  82.23525     82.23525     1900        47      48       69.90876
(392915,2)  82.23525     82.23525     1900        47      48       76.96519
(392915,4)  82.23525     82.23525     1950        52      53       80.18977
(392915,3)  82.23525     82.23525     1950        52      53       79.28574
(392915,2)  82.23525     82.23525     1950        52      53       81.85593
(392915,3)  82.23525     82.235